# **NLP701/NLP805@MBZUAI Fall 2024 - Lab 11**

## **Learning Outcomes**
- Critically analyze and evaluate, machine translation using seq2seq model.
- Gain proficiency in implementing/evaluating seq2seq model in machine translation.
- Identify important points in a research paper.

## **Learning Activities**
- Train a seq2seq model in Machine Translation.
- Evaluate the model using BLEU.
- Read/Skim a paper and discuss with classmates.

In [1]:
%matplotlib inline


# Machine Translation with a Sequence to Sequence Network
Heavily adapted from: [Sean Robertson](https://github.com/spro/practical-pytorch)

In this project we will be teaching a neural network to translate from
French to English.

    [KEY: > input, = target, < output]

    > il est en train de peindre un tableau .
    = he is painting a picture .
    < he is painting a picture .

    > pourquoi ne pas essayer ce vin delicieux ?
    = why not try that delicious wine ?
    < why not try that delicious wine ?

    > elle n est pas poete mais romanciere .
    = she is not a poet but a novelist .
    < she not not a poet but a novelist .

    > vous etes trop maigre .
    = you re too skinny .
    < you re all alone .

... to varying degrees of success.

This is made possible by the simple but powerful idea of the [sequence
to sequence network](https://arxiv.org/abs/1409.3215), in which two
recurrent neural networks work together to transform one sequence to
another. An encoder network condenses an input sequence into a vector,
and a decoder network unfolds that vector into a new sequence.

![picture](https://pytorch.org/tutorials/_images/seq2seq.png)

**Requirements**


In [2]:
from io import open
import unicodedata
import random
import time
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torch.utils.data import TensorDataset, DataLoader, RandomSampler


# utility functions we're going to use later
def asMinutes(s):
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)


def timeSince(since, percent):
    now = time.time()
    s = now - since
    es = s / (percent)
    rs = es - s
    return '%s (- %s)' % (asMinutes(s), asMinutes(rs))


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

/home/khang.nhat/anaconda3/envs/nlplab/lib/python3.11/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12090). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


device(type='cpu')

## Loading data files

The data for this project is a set of many thousands of English to
French translation pairs.

[This question on Open Data Stack
Exchange](https://opendata.stackexchange.com/questions/3888/dataset-of-sentences-translated-into-many-languages)_
pointed me to the open translation site https://tatoeba.org/ which has
downloads available at https://tatoeba.org/eng/downloads - and better
yet, someone did the extra work of splitting language pairs into
individual text files here: https://www.manythings.org/anki/

The English to French pairs are too big to include in the repo, so
download to ``data/eng-fra.txt`` before continuing. The file is a tab
separated list of translation pairs:


    I am cold.    J'ai froid.

Download the data and extract it to the current directory.



In [3]:
%%bash
wget https://download.pytorch.org/tutorial/data.zip
unzip data.zip

--2026-03-28 07:37:50--  https://download.pytorch.org/tutorial/data.zip
Resolving download.pytorch.org (download.pytorch.org)... 108.139.86.110, 108.139.86.17, 108.139.86.124, ...
Connecting to download.pytorch.org (download.pytorch.org)|108.139.86.110|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2882130 (2.7M) [application/zip]
Saving to: ‘data.zip’

     0K .......... .......... .......... .......... ..........  1%  548K 5s
    50K .......... .......... .......... .......... ..........  3%  550K 5s
   100K .......... .......... .......... .......... ..........  5% 8.32M 3s
   150K .......... .......... .......... .......... ..........  7%  590K 4s
   200K .......... .......... .......... .......... ..........  8% 9.01M 3s
   250K .......... .......... .......... .......... .......... 10% 10.2M 2s
   300K .......... .......... .......... .......... .......... 12% 15.4M 2s
   350K .......... .......... .......... .......... .......... 14%  652K 2s
   400K .

Archive:  data.zip
   creating: data/
  inflating: data/eng-fra.txt        
   creating: data/names/
  inflating: data/names/Arabic.txt   
  inflating: data/names/Chinese.txt  
  inflating: data/names/Czech.txt    
  inflating: data/names/Dutch.txt    
  inflating: data/names/English.txt  
  inflating: data/names/French.txt   
  inflating: data/names/German.txt   
  inflating: data/names/Greek.txt    
  inflating: data/names/Irish.txt    
  inflating: data/names/Italian.txt  
  inflating: data/names/Japanese.txt  
  inflating: data/names/Korean.txt   
  inflating: data/names/Polish.txt   
  inflating: data/names/Portuguese.txt  
  inflating: data/names/Russian.txt  
  inflating: data/names/Scottish.txt  
  inflating: data/names/Spanish.txt  
  inflating: data/names/Vietnamese.txt  


We will be representing each word in a language as a one-hot
vector, or giant vector of zeros except for a single one (at the index
of the word). We will however cheat a bit and trim the data to only use a few
thousand words per language.

![picture](https://pytorch.org/tutorials/_images/word-encoding.png)





We'll need a unique index per word to use as the inputs and targets of
the networks later. To keep track of all this we will use a helper class
called ``Lang`` which has word → index (``word2index``) and index → word
(``index2word``) dictionaries, as well as a count of each word
``word2count`` which will be used to replace rare words later.




In [4]:
SOS_token = 0
EOS_token = 1


class Lang:
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: "SOS", 1: "EOS", 2: "UNK"}
        self.n_words = 3  # Count SOS, EOS, and UNK

    def addSentence(self, sentence):
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

To normalize the input, we apply NFKC normalization, followed by lowercasing.
You may want to modify this function if you are working on other languages.




In [5]:
def normalizeString(s):
    s = unicodedata.normalize('NFKC', s)
    s = s.lower().strip()
    return s

To read the data file we will split the file into lines, and then split
lines into pairs.



In [6]:
def readLangs(lang1, lang2):
    print("Reading lines...")

    # Read the file and split into lines
    lines = open('./data/%s-%s.txt' % (lang1, lang2), encoding='utf-8').\
        read().strip().split('\n')

    # Split every line into pairs and normalize
    pairs = [[normalizeString(s) for s in l.split('\t')] for l in lines]

    pairs = [list(reversed(p)) for p in pairs]
    input_lang = Lang(lang2)
    output_lang = Lang(lang1)

    return input_lang, output_lang, pairs

Since there are a *lot* of example sentences and we want to train
something quickly, we'll trim the data set to only relatively short and
simple sentences.
Here the maximum length is 10 words (that includes
ending punctuation) and we're filtering to sentences that translate to
the form "I am" or "He is" etc. (accounting for apostrophes replaced
earlier).
This is English-specific so you need to define language-spepcific rules if you're working on other languages.



In [7]:
MAX_LENGTH = 10

eng_prefixes = (
    "i am ", "i m ",
    "he is", "he s ",
    "she is", "she s ",
    "you are", "you re ",
    "we are", "we re ",
    "they are", "they re "
)


def filterPair(p):
    return len(p[0].split(' ')) < MAX_LENGTH and \
        len(p[1].split(' ')) < MAX_LENGTH and \
        p[1].startswith(eng_prefixes)


def filterPairs(pairs):
    return [pair for pair in pairs if filterPair(pair)]

The full process for preparing the data is:

-  Read text file and split into lines, split lines into pairs
-  Normalize text, filter by length and content
-  Make word lists from sentences in pairs




In [8]:
def prepareData(lang1, lang2):
    input_lang, output_lang, pairs = readLangs(lang1, lang2)
    print("Read %s sentence pairs" % len(pairs))
    pairs = filterPairs(pairs)
    print("Trimmed to %s sentence pairs" % len(pairs))
    print("Counting words...")
    for pair in pairs:
        input_lang.addSentence(pair[0])
        output_lang.addSentence(pair[1])
    print("Counted words:")
    print(input_lang.name, input_lang.n_words)
    print(output_lang.name, output_lang.n_words)
    return input_lang, output_lang, pairs


input_lang, output_lang, pairs = prepareData('eng', 'fra')
train_pairs, test_pairs = pairs[-1000:], pairs[:-1000]
print(random.choice(train_pairs))

Reading lines...


Read 135842 sentence pairs
Trimmed to 2883 sentence pairs
Counting words...
Counted words:
fra 2783
eng 2050
['nous sommes confrontés à quantité de problèmes.', 'we are faced with a host of problems.']


## The Seq2Seq Model

A Recurrent Neural Network, or RNN, is a network that operates on a
sequence and uses its own output as input for subsequent steps.

A [Sequence to Sequence network](https://arxiv.org/abs/1409.3215)_, or
seq2seq network, or [Encoder Decoder
network](https://arxiv.org/pdf/1406.1078v3.pdf)_, is a model
consisting of two RNNs called the encoder and decoder. The encoder reads
an input sequence and outputs a single vector, and the decoder reads
that vector to produce an output sequence.

![picture](https://pytorch.org/tutorials/_images/seq2seq.png)

Unlike sequence prediction with a single RNN, where every input
corresponds to an output, the seq2seq model frees us from sequence
length and order, which makes it ideal for translation between two
languages.

Consider the sentence "Je ne suis pas le chat noir" → "I am not the
black cat". Most of the words in the input sentence have a direct
translation in the output sentence, but are in slightly different
orders, e.g. "chat noir" and "black cat". Because of the "ne/pas"
construction there is also one more word in the input sentence. It would
be difficult to produce a correct translation directly from the sequence
of input words.

With a seq2seq model the encoder creates a single vector which, in the
ideal case, encodes the "meaning" of the input sequence into a single
vector — a single point in some N dimensional space of sentences.




### The Encoder

The encoder of a seq2seq network is a RNN that outputs some value for
every word from the input sentence. For every input word the encoder
outputs a vector and a hidden state, and uses the hidden state for the
next input word.

![picture](https://pytorch.org/tutorials/_images/encoder-network.png)





In [9]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(input_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)

    def forward(self, input):
        embedded = self.embedding(input)
        output, hidden = self.gru(embedded)
        return output, hidden

### The Decoder

The decoder is another RNN that takes the encoder output vector(s) and
outputs a sequence of words to create the translation.




#### Simple Decoder

In the simplest seq2seq decoder we use only last output of the encoder.
This last output is sometimes called the *context vector* as it encodes
context from the entire sequence. This context vector is used as the
initial hidden state of the decoder.

At every step of decoding, the decoder is given an input token and
hidden state. The initial input token is the start-of-string ``<SOS>``
token, and the first hidden state is the context vector (the encoder's
last hidden state).

![picture](https://pytorch.org/tutorials/_images/decoder-network.png)





In [10]:
class DecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(DecoderRNN, self).__init__()
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)

    def forward(self, encoder_outputs, encoder_hidden, target_tensor=None):
        batch_size = encoder_outputs.size(0)
        decoder_input = torch.empty(batch_size, 1, dtype=torch.long, device=device).fill_(SOS_token)
        decoder_hidden = encoder_hidden
        decoder_outputs = []

        for i in range(MAX_LENGTH):
            decoder_output, decoder_hidden  = self.forward_step(decoder_input, decoder_hidden)
            decoder_outputs.append(decoder_output)

            if target_tensor is not None:
                # Teacher forcing: Feed the target as the next input
                decoder_input = target_tensor[:, i].unsqueeze(1) # Teacher forcing
            else:
                # Without teacher forcing: use its own predictions as the next input
                _, topi = decoder_output.topk(1)
                decoder_input = topi.squeeze(-1).detach()  # detach from history as input

        decoder_outputs = torch.cat(decoder_outputs, dim=1)
        decoder_outputs = F.log_softmax(decoder_outputs, dim=-1)
        return decoder_outputs, decoder_hidden, None # We return `None` for consistency in the training loop

    def forward_step(self, input, hidden):
        output = self.embedding(input)
        output = F.relu(output)
        output, hidden = self.gru(output, hidden)
        output = self.out(output)
        return output, hidden

## Training

### Preparing Training Data

To train, for each pair we will need an input tensor (indexes of the
words in the input sentence) and target tensor (indexes of the words in
the target sentence). While creating these vectors we will append the
EOS token to both sequences.




In [11]:
def indexesFromSentence(lang, sentence):
    sentence_ids = []
    for word in sentence.split(' '):
        if word in lang.word2index:
            sentence_ids.append(lang.word2index[word])
        else:
            sentence_ids.append(lang.word2index['UNK'])
    return sentence_ids


def tensorFromSentence(lang, sentence):
    indexes = indexesFromSentence(lang, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(1, -1)


def get_dataloader(batch_size):
    input_lang, output_lang, pairs = prepareData('eng', 'fra')
    train_pairs, test_pairs = pairs[-1000:], pairs[:-1000]

    n = len(train_pairs)
    input_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)
    target_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)

    for idx, (inp, tgt) in enumerate(train_pairs):
        inp_ids = indexesFromSentence(input_lang, inp)
        tgt_ids = indexesFromSentence(output_lang, tgt)
        inp_ids.append(EOS_token)
        tgt_ids.append(EOS_token)
        input_ids[idx, :len(inp_ids)] = inp_ids
        target_ids[idx, :len(tgt_ids)] = tgt_ids

    train_data = TensorDataset(torch.LongTensor(input_ids).to(device),
                               torch.LongTensor(target_ids).to(device))

    train_sampler = RandomSampler(train_data)
    train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)
    return input_lang, output_lang, train_dataloader

### Training the Model

To train we run the input sentence through the encoder, and keep track
of every output and the latest hidden state. Then the decoder is given
the ``<SOS>`` token as its first input, and the last hidden state of the
encoder as its first hidden state.

"Teacher forcing" is the concept of using the real target outputs as
each next input, instead of using the decoder's guess as the next input.
Using teacher forcing causes it to converge faster but [when the trained
network is exploited, it may exhibit
instability](http://citeseerx.ist.psu.edu/viewdoc/download?doi=10.1.1.378.4095&rep=rep1&type=pdf)_.

You can observe outputs of teacher-forced networks that read with
coherent grammar but wander far from the correct translation -
intuitively it has learned to represent the output grammar and can "pick
up" the meaning once the teacher tells it the first few words, but it
has not properly learned how to create the sentence from the translation
in the first place.

Because of the freedom PyTorch's autograd gives us, we can randomly
choose to use teacher forcing or not with a simple if statement. Turn
``teacher_forcing_ratio`` up to use more of it.




In [12]:
def train_epoch(dataloader, encoder, decoder, encoder_optimizer,
          decoder_optimizer, criterion):

    total_loss = 0
    for data in dataloader:
        input_tensor, target_tensor = data

        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, _, _ = decoder(encoder_outputs, encoder_hidden, target_tensor)

        loss = criterion(
            decoder_outputs.view(-1, decoder_outputs.size(-1)),
            target_tensor.view(-1)
        )
        loss.backward()

        encoder_optimizer.step()
        decoder_optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

The whole training process looks like this:

-  Start a timer
-  Initialize optimizers and criterion
-  Create set of training pairs
-  Start empty losses array for plotting

Then we call ``train`` many times and occasionally print the progress (%
of examples, time so far, estimated time) and average loss.

Plotting is done with matplotlib, using the array of loss values
``plot_losses`` saved while training.

In [13]:
import matplotlib.pyplot as plt
plt.switch_backend('agg')
import matplotlib.ticker as ticker
import numpy as np

def showPlot(points):
    plt.figure()
    fig, ax = plt.subplots()
    # this locator puts ticks at regular intervals
    loc = ticker.MultipleLocator(base=0.2)
    ax.yaxis.set_major_locator(loc)
    plt.plot(points)

In [14]:
def train(train_dataloader, encoder, decoder, n_epochs, learning_rate=0.001,
               print_every=100, plot_every=100):
    start = time.time()
    plot_losses = []
    print_loss_total = 0  # Reset every print_every
    plot_loss_total = 0  # Reset every plot_every

    encoder_optimizer = optim.Adam(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.Adam(decoder.parameters(), lr=learning_rate)
    criterion = nn.NLLLoss()

    for epoch in range(1, n_epochs + 1):
        loss = train_epoch(train_dataloader, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion)
        print_loss_total += loss
        plot_loss_total += loss

        if epoch % print_every == 0:
            print_loss_avg = print_loss_total / print_every
            print_loss_total = 0
            print('%s (%d %d%%) %.4f' % (timeSince(start, epoch / n_epochs),
                                        epoch, epoch / n_epochs * 100, print_loss_avg))

        if epoch % plot_every == 0:
            plot_loss_avg = plot_loss_total / plot_every
            plot_losses.append(plot_loss_avg)
            plot_loss_total = 0

    showPlot(plot_losses)

Now we can actually initialize a network and start training.

Remember that the input sentences were heavily filtered. For this small
dataset we can use relatively small networks of 256 hidden nodes and a
single GRU layer.

In [15]:
hidden_size = 256
batch_size = 32

input_lang, output_lang, train_dataloader = get_dataloader(batch_size)

encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder = DecoderRNN(hidden_size, output_lang.n_words).to(device)

train(train_dataloader, encoder, decoder, 80, print_every=5, plot_every=5)

Reading lines...


Read 135842 sentence pairs
Trimmed to 2883 sentence pairs
Counting words...
Counted words:
fra 2783
eng 2050
0m 3s (- 0m 51s) (5 6%) 3.4063
0m 6s (- 0m 43s) (10 12%) 1.9275
0m 9s (- 0m 39s) (15 18%) 1.1422
0m 12s (- 0m 36s) (20 25%) 0.6667
0m 14s (- 0m 32s) (25 31%) 0.3820
0m 17s (- 0m 29s) (30 37%) 0.2141
0m 20s (- 0m 26s) (35 43%) 0.1180
0m 23s (- 0m 23s) (40 50%) 0.0661
0m 26s (- 0m 20s) (45 56%) 0.0416
0m 29s (- 0m 17s) (50 62%) 0.0290
0m 32s (- 0m 14s) (55 68%) 0.0220
0m 34s (- 0m 11s) (60 75%) 0.0176
0m 37s (- 0m 8s) (65 81%) 0.0147
0m 40s (- 0m 5s) (70 87%) 0.0125
0m 43s (- 0m 2s) (75 93%) 0.0114
0m 46s (- 0m 0s) (80 100%) 0.0097


Remember that the input sentences were heavily filtered. For this small
dataset we can use relatively small networks of 256 hidden nodes and a
single GRU layer.

## Prediction

Prediction is mostly the same as training, but there are no targets so
we simply feed the decoder's predictions back to itself for each step.
Every time it predicts a word we add it to the output string, and if it
predicts the EOS token we stop there.

In [16]:
def predict(encoder, decoder, sentence, input_lang, output_lang):
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)
        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, decoder_hidden, _ = decoder(encoder_outputs, encoder_hidden)
        _, topi = decoder_outputs.topk(1)
        decoded_ids = topi.squeeze()

        decoded_words = []
        for idx in decoded_ids:
            if idx.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            decoded_words.append(output_lang.index2word[idx.item()])
    return decoded_words, _

We can evaluate random sentences from the training set and print out the
input, target, and output to make some subjective quality judgements:

In [17]:
def evaluateRandomly(encoder, decoder, n=10):
    for i in range(n):
        pair = random.choice(pairs)
        print('>', pair[0])
        print('=', pair[1])
        output_words, _ = predict(encoder, decoder, pair[0], input_lang, output_lang)
        output_sentence = ' '.join(output_words)
        print('<', output_sentence)
        print('')

In [18]:
evaluateRandomly(encoder, decoder)

> il a plutôt raison.
= he is quite right.
< he is absorbed in reading a detective story. <EOS>

> elle est à son hôtel, maintenant.
= she is in her hotel now.
< she is used to staying up all night. <EOS>

> tu as toujours à te plaindre.
= you are always complaining.
< you are always doubting my word. <EOS>

> c'est que nous appelons un pionnier.
= he is what we call a pioneer.
< he is going to drive a great day today. <EOS>

> en un sens, vous avez raison.
= you are right in a way.
< you are so childish sometimes. <EOS>

> vous arrivez trop tard.
= you are too late.
< you are too critical of others' shortcomings. <EOS>

> il est dingue de jazz.
= he is crazy about jazz.
< he is used to such situations. <EOS>

> il a honte de poser des questions.
= he is ashamed to ask questions.
< he is ashamed to ask questions. <EOS>

> nous sommes ici.
= we are here.
< we are going to visit our aunt next sunday. <EOS>

> j'ai peur des ours.
= i am afraid of bears.
< i am afraid of having trouble. <E

# **Exercise: Evaluate with BLEU**

Your task is to evalute the model on test set (`test_pairs`) using **BLEU-1, BLEU-2, BLEU-4**.
- Predict translations of the source sentences in the test (predictions/system outputs).
- Extract the target translations (references).
- Find an appropriate tool to compute BLEU (e.g., `sacrebleu`, `nltk`, `evaluate`)

Reference: https://aclanthology.org/P02-1040/
 - NAACL 2018 Test-of-Time Award Paper


In [19]:
import nltk

# Ensure BLEU scorer is available
try:
    from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
except ImportError:
    raise ImportError("Please install nltk: pip install nltk")

smoothie = SmoothingFunction().method4

# Predict translations for test set
system_outputs = []
references = []

for pair in test_pairs:
    src_sentence = pair[0]
    tgt_sentence = pair[1]
    output_words, _ = predict(encoder, decoder, src_sentence, input_lang, output_lang)
    # Remove <EOS> and any tokens after it
    if '<EOS>' in output_words:
        output_words = output_words[:output_words.index('<EOS>')]
    system_outputs.append(output_words)
    # Reference: list of list of tokens
    references.append([tgt_sentence.split()])

# Compute BLEU-1, BLEU-2, BLEU-4
bleu1 = corpus_bleu(references, system_outputs, weights=(1, 0, 0, 0), smoothing_function=smoothie)
bleu2 = corpus_bleu(references, system_outputs, weights=(0.5, 0.5, 0, 0), smoothing_function=smoothie)
bleu4 = corpus_bleu(references, system_outputs, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoothie)

print(f"BLEU-1: {bleu1:.4f}")
print(f"BLEU-2: {bleu2:.4f}")
print(f"BLEU-4: {bleu4:.4f}")


BLEU-1: 0.3343
BLEU-2: 0.2579
BLEU-4: 0.0845
